# Sequential Workflow with LangGraph

A three-stage content pipeline built as a **sequential graph** in LangGraph.

Raw, messy text goes in at one end. A polished, localized Hinglish video script
comes out the other. Each stage does exactly one job and hands its result to the next:

```
START -> editor -> scriptwriter -> translator -> END
```

| Stage | Node | What it does |
|---|---|---|
| 1 | `editor` | Fixes grammar, spelling and tone |
| 2 | `scriptwriter` | Rewrites the clean text as a punchy video hook |
| 3 | `translator` | Localizes the hook into natural Hinglish |

The point of the notebook is the **pattern**, not the prompts: how shared state flows
through a graph, and how each node reads one key and writes another.

---

## 1. Setup

Install LangGraph (the graph runtime) and LangChain (the LLM abstractions on top of it).

In [23]:
!pip install -U langgraph langchain

`langchain-groq` is the provider binding. Groq serves open models at very high token
throughput, which keeps a three-hop chain like this feeling instant.

In [24]:
%pip install -U langchain-groq

Quick sanity check that the provider package actually landed and which version we are on.

In [25]:
!pip show langchain-groq

Name: langchain-groq
Version: 1.1.3
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: groq, langchain-core
Required-by: 


## 2. Imports

`TypedDict` is the important one — it is what LangGraph uses to describe the shape of
the state that travels through the graph.

In [26]:
import os
from typing import TypedDict


## 3. Defining the State

The state is the **single object passed from node to node**. Every node receives the whole
state and returns a partial dict; LangGraph merges that return value back in.

Notice how the four keys map onto the pipeline:

- `raw_input` — what the user supplies
- `edited_text` — written by the editor, read by the scriptwriter
- `script_text` — written by the scriptwriter, read by the translator
- `final_output` — written by the translator, returned to the caller

Each stage only touches the key it owns. That is what makes the nodes swappable.

In [27]:
#creation of state
class pipelineSate(TypedDict):
  raw_input : str
  edited_text : str
  script_text : str
  final_output : str


## 4. The Model

One `ChatGroq` client is shared by all three nodes. The API key is pulled from Colab's
secret store rather than hardcoded, so the notebook can be committed safely.

`temperature=0.8` is deliberately high — two of the three stages are creative rewriting,
and a low temperature makes them read flat.

> **Running outside Colab?** Replace the `userdata.get(...)` line with
> `os.environ["GROQ_API_KEY"]` and export the key in your shell.

In [36]:
#we created our llm
from langchain_groq import ChatGroq
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.8,
    groq_api_key=groq_api_key
)

## 5. Stage 1 — The Editor Node

A node is just a function: `state in, partial state out`.

This one reads `raw_input`, asks the model to clean up grammar and transitions without
changing the meaning, and returns `edited_text`. Nothing else in the state is touched.

Returning `{"edited_text": ...}` rather than the full state is the idiomatic LangGraph
style — the runtime handles the merge.

In [29]:
#creating the editor Node
def editor_node(state : pipelineSate) -> dict:
  """stage 1 : clearns up grammar and removes typos and refines the tone """

  prompt = (
    "You are an expert copyeditor. Clean up the following raw text. "
    "Fix any grammatical errors, spelling mistakes, and smooth out the transitions "
    "while keeping the core message intact. Return only the edited text.\n\n"
    f"Text:\n{state['raw_input']}"
  )

  response = llm.invoke(prompt)

  return{"edited_text" : response.content.strip()}


## 6. Stage 2 — The Scriptwriter Node

Now that the text is clean, this node changes its *form*: prose becomes a spoken,
conversational video hook.

It reads `edited_text` and writes `script_text`. It never sees `raw_input`, and that
separation is intentional — the scriptwriter should not be re-fixing typos.

In [38]:
#creating scriptWriter Node
def scriptwriter_node(state : pipelineSate) -> dict:
  """Stage 2 : Fromat the clean text into an engaging video scirpt style."""

  print("\n--[Stage 2] Executing Scriptwriter Node ---")

  prompt = (
    "You are a charismatic YouTube content creator. Take this edited text and transform "
    "it into a highly engaging, punchy, conversational video script hook. Make it sound "
    "like a real person speaking passionately. Return only the script content.\n\n"
    f"Edited Text:\n{state['edited_text']}"
  )

  response = llm.invoke(prompt)
  return {"script_text" : response.content.strip()}


## 7. Stage 3 — The Translator Node

The final stage localizes the script into Hinglish.

The prompt is the most heavily constrained of the three, and for good reason: asked to
"translate to Hinglish", models tend to drift into pure Hindi. So it explicitly instructs
the model to keep technical vocabulary (AI, coding, data, productivity) in English and mix
Hindi phrasing around it — the way an Indian tech creator actually speaks.

Reads `script_text`, writes `final_output`.

In [41]:
#translater node
def translator_node(state: pipelineSate) -> dict:
    """Stage 3: Translates the script into natural flowing Hinglish."""

    print("\n--- [Stage 3] Executing Hinglish Translator Node ---")

    prompt = (
    "You are an expert content localizer for the Indian market. "
    "Convert the following English script into natural conversational Hinglish. "
    "Hinglish means a natural MIX of Hindi and English, like how an Indian "
    "tech creator would actually speak. Do NOT translate the entire script into Hindi. "
    "Keep technical terms such as AI, technology, coding, productivity, data, etc. "
    "in English where they sound natural. Mix Hindi phrases naturally with English "
    "sentences. The final output should contain BOTH Hindi and English. "
    "Do not use pure Hindi throughout. "
    "Return only the final Hinglish script.\n\n"
    f"Script:\n{state['script_text']}"
    )

    response = llm.invoke(prompt)

    return {
        "final_output": response.content.strip()
    }



## 8. Wiring the Graph

`StateGraph` is the builder, and `START` / `END` are the sentinel nodes marking the
entry and exit points of the pipeline.

In [32]:
from langgraph.graph import StateGraph, START, END

## 9. Building, Compiling and Running

Three steps happen here:

1. **Register the nodes** with `add_node(name, function)`.
2. **Connect them** with `add_edge`. Because every edge is unconditional, execution is
   strictly linear — no branching, no loops. This is the sequential pattern in its
   simplest form.
3. **Compile** with `graph.compile()`, which validates the wiring and returns a runnable
   app exposing `.invoke()`.

`invoke` is given only `raw_input`; the remaining three keys are filled in by the nodes as
the state travels through the graph.

In [42]:
#create graph
graph = StateGraph(pipelineSate)

#add nodes to the graph
graph.add_node("editor", editor_node)
graph.add_node("scriptwriter", scriptwriter_node)
graph.add_node("translator", translator_node)

#add edges (sequential one after another)
graph.add_edge(START, "editor")
graph.add_edge("editor", "scriptwriter")
graph.add_edge("scriptwriter", "translator")
graph.add_edge("translator", END)


#compile the graph
app = graph.compile()

result = app.invoke({
    "raw_input" : "Artificial intelligence is changing the way we work, learn, and communicate. But the real question is not whether AI will replace humans. The bigger question is how humans can use AI to become more productive, creative, and efficient. From writing code to analyzing data and creating content, AI is becoming a powerful tool in almost every industry. The people who learn how to work alongside AI will have a major advantage in the future."
})

print("your result are : - \n\n")
print(result['final_output'])




--[Stage 2] Executing Scriptwriter Node ---

--- [Stage 3] Executing Hinglish Translator Node ---
your result are : - 


Hey, kya haal hai, fam? 🎉 Socho ek aisa world jahan tumhare paas ek super‑smart sidekick ho jo code likh sakta hai, numbers ko crunch kar sakta hai, aur seconds mein killer content generate kar deta hai. Yeh hi AI aaj ka scene hai—aur yeh humare jobs ko chheen ne ke liye nahi, balki hume level‑up karne ke liye aaya hai!  

Ab socho, data ke ocean mein dubne ki jagah tum usko ninja ki tarah slice‑kar rahe ho. Blank screen ko stare‑kar stress lene ki jagah, tumhare paas ek creative partner hai jo ideas tumhare saamne itni fast throw karta hai ki tum “brainstorm” bolte hi uska answer mil jaata hai.  

Real question yeh nahi ki “AI humko replace karega?”—balki “Hum AI ke saath kaise team up kar ke zyada productive, zyada creative, aur unstoppable ban sakte hain?” Jo log is wave ko ride karna seekhenge, wahi future ko own karenge. Toh seat belt baandh lo, kyunki hum abhi

---

## Where to Take This Next

The sequential shape is the foundation. Natural extensions:

- **Conditional edges** — route to different nodes based on state (e.g. skip translation
  if the target audience is English).
- **Parallel fan-out** — generate several script variants at once and pick the best.
- **Checkpointing** — persist state so a run can be paused, inspected and resumed.
- **Human-in-the-loop** — interrupt before the translator and let a person approve the script.

Swapping the prompts turns this same graph into any three-stage pipeline: summarize ->
outline -> draft, extract -> validate -> format, and so on. The wiring does not change.